# 01 — Exploratory Data Analysis

**Auralytics** | CPEN 355 Final Project

This notebook inspects the DCASE 2020 Task 2 dataset before any training:
- Dataset structure and clip counts
- Waveform visualization
- Log-mel spectrogram visualization (normal vs anomalous)
- Spectrogram statistics across machine types

Run `python src/preprocess.py` first to generate `data/processed/`.

In [ ]:
import sys
sys.path.append('..')  # allow imports from src/

from pathlib import Path
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.dataset import DCASEDataset
from src.preprocess import load_audio, compute_log_mel, normalize
from src.utils import plot_spectrogram

plt.rcParams['figure.facecolor'] = '#0d0d0d'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333'
plt.rcParams['text.color']       = '#e8e8e8'
plt.rcParams['axes.labelcolor']  = '#e8e8e8'
plt.rcParams['xtick.color']      = '#888'
plt.rcParams['ytick.color']      = '#888'

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
MACHINE_TYPES = ['fan', 'pump', 'valve']

## 1. Dataset Inventory
Count clips per machine type and split.

In [ ]:
print(f"{'Machine':<10} {'Split':<8} {'Normal':>8} {'Anomalous':>10} {'Total':>7}")
print('─' * 48)

for machine in MACHINE_TYPES:
    for split in ['train', 'test']:
        split_dir = RAW_DIR / machine / split
        if not split_dir.exists():
            print(f"{machine:<10} {split:<8} {'NOT FOUND':>25}")
            continue
        wavs      = list(split_dir.glob('*.wav'))
        normal    = sum(1 for f in wavs if f.stem.startswith('normal'))
        anomalous = sum(1 for f in wavs if f.stem.startswith('anomaly'))
        print(f"{machine:<10} {split:<8} {normal:>8} {anomalous:>10} {len(wavs):>7}")

## 2. Waveform Inspection
Listen to what normal vs anomalous audio looks like in the time domain.

In [ ]:
MACHINE = 'fan'   # change to 'pump' or 'valve' to explore others

test_dir    = RAW_DIR / MACHINE / 'test'
normal_wav  = next(test_dir.glob('normal*.wav'),   None)
anomaly_wav = next(test_dir.glob('anomaly*.wav'),  None)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle(f'{MACHINE.upper()} — Waveform Comparison', fontsize=14, fontweight='bold')

for ax, path, label, color in zip(
    axes,
    [normal_wav, anomaly_wav],
    ['Normal', 'Anomalous'],
    ['#34d399', '#ef4444'],
):
    if path is None:
        ax.set_title(f'{label} — file not found')
        continue
    wave, sr = librosa.load(str(path), sr=16000, mono=True, duration=10.0)
    t = np.linspace(0, len(wave) / sr, len(wave))
    ax.plot(t, wave, color=color, linewidth=0.5, alpha=0.9)
    ax.set_ylabel('Amplitude')
    ax.set_title(label, color=color, fontweight='bold')
    ax.set_ylim(-0.5, 0.5)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 3. Spectrogram Comparison
This is what the model actually sees — log-mel spectrograms.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'{MACHINE.upper()} — Log-Mel Spectrogram Comparison', fontsize=14, fontweight='bold')

for ax, path, label, color in zip(
    axes,
    [normal_wav, anomaly_wav],
    ['Normal', 'Anomalous'],
    ['#34d399', '#ef4444'],
):
    if path is None:
        ax.set_title('file not found'); continue
    wave = load_audio(path)
    spec = compute_log_mel(wave)            # raw dB spectrogram
    im   = ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(label, color=color, fontweight='bold')
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Mel bins')
    plt.colorbar(im, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 4. All Three Machine Types — Normal Spectrograms
See how different each machine looks so we understand why we train per machine type.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Normal Spectrograms Across Machine Types', fontsize=14, fontweight='bold')

for ax, machine in zip(axes, MACHINE_TYPES):
    wav_path = next((RAW_DIR / machine / 'test').glob('normal*.wav'), None)
    if wav_path is None:
        wav_path = next((RAW_DIR / machine / 'train').glob('normal*.wav'), None)
    if wav_path is None:
        ax.set_title(f'{machine} — not found'); continue
    wave = load_audio(wav_path)
    spec = compute_log_mel(wave)
    im   = ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(machine.upper(), fontweight='bold')
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Mel bins')
    plt.colorbar(im, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 5. Processed Dataset — Quick Sanity Check
Confirm the preprocessed `.npy` files loaded correctly through our Dataset class.

In [ ]:
print('Processed dataset summary\n' + '─' * 40)
for machine in MACHINE_TYPES:
    for split in ['train', 'test']:
        try:
            ds = DCASEDataset(PROCESSED_DIR, machine, split)
            spec, label = ds[0]
            print(f"  {machine}/{split:<6} | {repr(ds)}")
            print(f"             | spec shape: {tuple(spec.shape)} | dtype: {spec.dtype}")
        except FileNotFoundError:
            print(f"  {machine}/{split:<6} | not preprocessed yet — run src/preprocess.py")

## ✅ EDA Complete

Key takeaways:
- Each machine type has a visually distinct spectral signature
- Anomalous clips show measurable differences in frequency distribution
- The dataset is clean with consistent clip length (10s, 16kHz, mono)
- Train split is all-normal — perfect for unsupervised autoencoder training

**Next:** `02_preprocessing.ipynb` or jump straight to `03_training.ipynb`